# SGLang RadixAttention 功能使用 — 极简 Demo

**定位**：演示 SGLang 独有的 **RadixAttention**（基数树前缀缓存）机制，通过对比「共享前缀」与「不共享前缀」场景下的推理速度，直观感受 KV 缓存复用带来的加速效果。

> **默认启动命令：**
> ```bash
> python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
> ```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台。*

---

## 硬件与环境要求

| 项目 | 说明 |
|------|------|
| **GPU** | NVIDIA GPU（算力 ≥ 7.0），单卡即可 |
| **显存（VRAM）** | ≥ 6 GB（Qwen2.5-0.5B-Instruct） |
| **驱动** | NVIDIA Driver ≥ 525 |
| **CUDA** | ≥ 12.1 |
| **Python** | 3.9 – 3.12 |
| **操作系统** | Linux x86_64 最佳；Windows 建议使用 WSL2 |

---

## 依赖安装

```bash
# 0) 创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate

# 1) 升级 pip
python -m pip install --upgrade pip

# 2) 安装 PyTorch（CUDA 12.4 示例）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3) 安装 SGLang 全部组件
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"

# 4) Jupyter
pip install notebook ipykernel
```

---

## 使用指南

1. 按顺序执行每个代码单元格。
2. 先启动 SGLang 服务（RadixAttention 默认开启，无需额外参数）。
3. 依次运行「共享前缀」和「无共享前缀」两组实验，观察耗时差异。
4. 查看「结果对比」单元格中的汇总数据。
5. 实验完成后务必运行「清理资源」释放 GPU 显存。

---

In [ ]:
# === 环境检查 ===

import sys  # 导入 sys 获取 Python 版本信息

print("Python 版本:", sys.version)  # 打印 Python 版本

try:
    import torch  # 尝试导入 PyTorch
except ImportError:
    print("未检测到 torch：请先安装带 CUDA 的 PyTorch。")  # 提示安装
else:
    print("torch 版本:", torch.__version__)  # 打印 torch 版本
    print("CUDA 是否可用:", torch.cuda.is_available())  # 检测 GPU
    if torch.cuda.is_available():  # 有 GPU 时打印详情
        print("GPU0 名称:", torch.cuda.get_device_name(0))  # GPU 产品名
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)  # 计算显存 GB
        print("GPU0 显存(GB):", vram_gb)  # 打印显存

In [ ]:
# 可选：在 Notebook 内核中安装依赖

import subprocess  # 导入 subprocess 用于执行 pip
import sys  # 导入 sys 获取解释器路径

_packages = ["sglang[all]", "openai>=1.0.0", "requests>=2.28.0"]  # 需要安装的包
_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "pip"] + _packages  # 组装安装命令
subprocess.run(_cmd, check=True)  # 执行安装
print("依赖安装完毕。")  # 提示完成

## 什么是 RadixAttention？

**RadixAttention** 是 SGLang 的核心创新之一，它使用 **基数树（Radix Tree）** 数据结构来自动管理 KV 缓存的复用。

### 工作原理

1. 当多个请求共享 **相同的前缀**（如相同的系统提示词 / System Prompt）时，SGLang 会自动检测这些共享部分。
2. 第一次请求会正常计算完整的 KV 张量并缓存到基数树中。
3. 后续请求如果前缀匹配，SGLang 会 **直接复用已缓存的 KV 张量**，跳过前缀部分的重复计算。
4. 这种机制对于 **多轮对话**、**批量请求使用相同系统提示词** 等场景尤其有效。

### 预期效果

- 共享前缀越长、复用次数越多，加速越明显
- 第 2 次及后续请求的延迟应低于第 1 次（首次需要完整计算并缓存）
- SGLang 启动时 **默认开启** RadixAttention，无需额外配置

---

## 启动 SGLang 服务

RadixAttention 在 SGLang 中 **默认启用**，无需额外参数。

---

In [ ]:
# 启动 SGLang 推理服务（RadixAttention 默认开启）

import json  # 用于解析 JSON 响应
import os  # 用于进程组操作
import signal  # 用于发送终止信号
import subprocess  # 用于创建子进程
import sys  # 获取 Python 解释器路径
import time  # 用于计时和等待
import urllib.error  # 捕获 HTTP 异常
import urllib.request  # 发送 HTTP 请求

HOST = "127.0.0.1"  # 监听地址
PORT = 30000  # 监听端口
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 模型 ID

_sglang_process = None  # 全局进程句柄

cmd = [  # 组装 SGLang 启动命令
    sys.executable, "-m", "sglang.launch_server",  # 以模块方式启动 SGLang
    "--model-path", MODEL_ID,  # 指定模型路径
    "--host", HOST,  # 绑定监听地址
    "--port", str(PORT),  # 绑定监听端口
]

print("启动命令:", " ".join(cmd))  # 打印完整命令便于复制

_sglang_process = subprocess.Popen(  # 以后台子进程启动服务
    cmd,
    stdout=subprocess.DEVNULL,  # 丢弃标准输出
    stderr=subprocess.STDOUT,  # 合并错误输出
    start_new_session=True,  # 创建新进程组
)

models_url = f"http://{HOST}:{PORT}/v1/models"  # 探活端点
deadline = time.time() + 600  # 最长等待 600 秒
print("正在等待 SGLang 服务就绪...")  # 提示用户等待

while time.time() < deadline:  # 轮询等待
    if _sglang_process.poll() is not None:  # 检查进程是否异常退出
        raise RuntimeError(f"SGLang 进程已退出，退出码: {_sglang_process.returncode}")  # 报错
    try:
        with urllib.request.urlopen(models_url, timeout=2) as resp:  # 探活请求
            json.loads(resp.read().decode("utf-8"))  # 解析确认有效
        print("SGLang 服务已就绪！RadixAttention 默认开启。")  # 确认就绪
        break  # 退出等待循环
    except Exception:  # 连接失败继续等待
        time.sleep(2)  # 等待 2 秒
else:
    raise TimeoutError("等待 SGLang 就绪超时（600s）")  # 超时报错

## 演示：共享前缀加速

使用 **相同的长系统提示词** 发送多次请求，观察第 2、3 次请求是否因为前缀缓存命中而更快。

---

In [ ]:
# 演示共享前缀加速：多次请求使用相同的长系统提示词

from openai import OpenAI  # 导入 OpenAI SDK
import time  # 导入 time 用于计时

client = OpenAI(  # 创建客户端
    base_url=f"http://{HOST}:{PORT}/v1",  # 指向本地 SGLang 服务
    api_key="EMPTY",  # 本地无需真实密钥
)

SHARED_SYSTEM_PROMPT = (  # 定义一个较长的共享系统提示词，增加前缀缓存的收益
    "你是一位资深的人工智能技术顾问，拥有超过20年的行业经验。"
    "你精通深度学习、自然语言处理、计算机视觉、强化学习等多个领域。"
    "你曾在多家顶级科技公司和研究机构工作，参与过多个大规模AI系统的设计与部署。"
    "你的回答应当专业、准确、有深度，同时能够用通俗易懂的方式解释复杂概念。"
    "请始终以中文回答，并在需要时提供具体的技术细节和实际案例。"
    "如果问题超出你的专业范围，请诚实说明并给出可能的方向建议。"
)

user_queries = [  # 定义三个不同的用户问题（系统提示词相同）
    "Transformer 架构的核心创新是什么？",  # 第 1 个问题
    "大语言模型的推理优化有哪些主要方向？",  # 第 2 个问题
    "KV Cache 在推理过程中起什么作用？",  # 第 3 个问题
]

shared_prefix_times = []  # 存储每次请求的耗时

print("=== 共享前缀测试（相同 System Prompt）===")  # 测试标题
print(f"System Prompt 长度: {len(SHARED_SYSTEM_PROMPT)} 字符\n")  # 显示前缀长度

for i, query in enumerate(user_queries):  # 遍历每个问题
    t0 = time.time()  # 记录请求开始时间
    completion = client.chat.completions.create(  # 发送聊天补全请求
        model=MODEL_ID,  # 指定模型
        messages=[  # 消息列表
            {"role": "system", "content": SHARED_SYSTEM_PROMPT},  # 共享的系统提示词
            {"role": "user", "content": query},  # 不同的用户问题
        ],
        temperature=0.3,  # 较低温度
        max_tokens=100,  # 限制生成长度
    )
    elapsed = time.time() - t0  # 计算耗时
    shared_prefix_times.append(elapsed)  # 记录耗时
    reply = completion.choices[0].message.content  # 获取回复内容
    tokens = completion.usage.completion_tokens  # 获取生成 token 数
    print(f"请求 {i+1}: 耗时 {elapsed:.3f}s | 生成 {tokens} tokens | 问题: {query}")  # 打印结果
    print(f"  回复: {reply[:80]}...\n")  # 打印回复前 80 字符

print("共享前缀各请求耗时:", [f"{t:.3f}s" for t in shared_prefix_times])  # 汇总显示所有耗时

## 对比：无共享前缀

每次请求使用 **完全不同的系统提示词**，使得前缀缓存无法命中，观察耗时是否一直较慢。

---

In [ ]:
# 对比实验：每次使用不同的系统提示词，无法触发前缀缓存

different_system_prompts = [  # 三个完全不同的长系统提示词
    (  # 第 1 个：医学领域专家
        "你是一位经验丰富的医学研究专家，专注于分子生物学和基因组学研究。"
        "你在全球顶级医学期刊上发表过数百篇论文，对疾病机理和药物研发有深入理解。"
        "你善于将复杂的医学概念用简明的语言解释给非专业人士，同时保持科学的严谨性。"
        "请始终以中文回答，并注意区分已证实的科学结论和尚在研究中的假说。"
        "在涉及健康建议时，请提醒读者咨询专业医生。"
    ),
    (  # 第 2 个：金融分析师
        "你是一位资深的全球金融市场分析师，拥有CFA和FRM双重认证。"
        "你精通股票、债券、外汇、衍生品等多个金融市场的分析方法和投资策略。"
        "你曾在多家国际投行和对冲基金工作，管理过数十亿美元规模的投资组合。"
        "你的分析以数据驱动、逻辑清晰著称，善于从宏观经济和微观财务两个维度进行研判。"
        "请始终以中文回答，并在涉及投资建议时注明相关风险提示。"
    ),
    (  # 第 3 个：教育专家
        "你是一位从事教育研究三十年的资深教育学教授，专注于认知科学与学习理论。"
        "你对全球各国的教育体系和教学方法有广泛的研究和实践经验。"
        "你擅长将教育理论与实际教学场景结合，提出切实可行的教学改进方案。"
        "你特别关注技术驱动的教育创新，包括在线学习、自适应学习和AI辅助教育。"
        "请始终以中文回答，结合中国教育的实际情况给出针对性建议。"
    ),
]

different_queries = [  # 对应的用户问题
    "基因编辑技术的最新进展是什么？",  # 医学问题
    "如何评估一家科技公司的长期投资价值？",  # 金融问题
    "如何提高大学课堂的教学效果？",  # 教育问题
]

no_shared_times = []  # 存储每次请求的耗时

print("=== 无共享前缀测试（不同 System Prompt）===")  # 测试标题

for i in range(len(different_system_prompts)):  # 遍历每组提示词
    t0 = time.time()  # 记录开始时间
    completion = client.chat.completions.create(  # 发送请求
        model=MODEL_ID,  # 指定模型
        messages=[  # 消息列表
            {"role": "system", "content": different_system_prompts[i]},  # 每次不同的系统提示词
            {"role": "user", "content": different_queries[i]},  # 对应的问题
        ],
        temperature=0.3,  # 保持与共享测试一致的参数
        max_tokens=100,  # 限制生成长度
    )
    elapsed = time.time() - t0  # 计算耗时
    no_shared_times.append(elapsed)  # 记录耗时
    reply = completion.choices[0].message.content  # 获取回复
    tokens = completion.usage.completion_tokens  # 获取 token 数
    print(f"请求 {i+1}: 耗时 {elapsed:.3f}s | 生成 {tokens} tokens | 问题: {different_queries[i]}")  # 打印结果
    print(f"  回复: {reply[:80]}...\n")  # 打印回复前 80 字符

print("无共享前缀各请求耗时:", [f"{t:.3f}s" for t in no_shared_times])  # 汇总显示所有耗时

## 结果对比

---

In [ ]:
# 汇总对比共享前缀 vs 无共享前缀的测试结果

print("=" * 65)  # 打印分隔线
print("RadixAttention 前缀缓存效果对比")  # 标题
print("=" * 65)  # 打印分隔线

print(f"\n{'场景':<20} {'请求1':>10} {'请求2':>10} {'请求3':>10} {'平均':>10}")  # 表头
print("-" * 65)  # 分隔线

shared_avg = sum(shared_prefix_times) / len(shared_prefix_times)  # 计算共享前缀平均耗时
no_shared_avg = sum(no_shared_times) / len(no_shared_times)  # 计算无共享前缀平均耗时

shared_strs = [f"{t:.3f}s" for t in shared_prefix_times]  # 格式化共享前缀各耗时
no_shared_strs = [f"{t:.3f}s" for t in no_shared_times]  # 格式化无共享前缀各耗时

print(f"{'共享前缀':<20} {shared_strs[0]:>10} {shared_strs[1]:>10} {shared_strs[2]:>10} {shared_avg:>9.3f}s")  # 打印共享前缀行
print(f"{'无共享前缀':<20} {no_shared_strs[0]:>10} {no_shared_strs[1]:>10} {no_shared_strs[2]:>10} {no_shared_avg:>9.3f}s")  # 打印无共享前缀行

print("-" * 65)  # 分隔线

if shared_prefix_times[0] > 0:  # 避免除以零
    speedup_2nd = shared_prefix_times[0] / shared_prefix_times[1] if shared_prefix_times[1] > 0 else 0  # 计算第 2 次相对第 1 次的加速
    speedup_3rd = shared_prefix_times[0] / shared_prefix_times[2] if shared_prefix_times[2] > 0 else 0  # 计算第 3 次相对第 1 次的加速
    print(f"\n共享前缀场景中：")  # 小结标题
    print(f"  第 2 次请求相对第 1 次加速: {speedup_2nd:.2f}x")  # 打印加速比
    print(f"  第 3 次请求相对第 1 次加速: {speedup_3rd:.2f}x")  # 打印加速比

if no_shared_avg > 0:  # 避免除以零
    overall_ratio = no_shared_avg / shared_avg if shared_avg > 0 else 0  # 计算整体比值
    print(f"\n无共享前缀平均耗时 / 共享前缀平均耗时 = {overall_ratio:.2f}x")  # 打印整体对比

print("\n" + "=" * 65)  # 结束分隔线
print("提示：0.5B 小模型上效果可能不显著，更大模型 + 更长前缀效果更明显。")  # 说明预期

## 清理资源

---

In [ ]:
# 停止 SGLang 子进程，释放 GPU 显存

import os  # 导入 os 用于进程组操作
import signal  # 导入 signal 用于发送终止信号

try:
    _sglang_process  # 检查变量是否存在
except NameError:
    print("未找到 _sglang_process：若服务在终端启动，请手动 Ctrl+C。")  # 提示用户
else:
    if _sglang_process is None or _sglang_process.poll() is not None:  # 进程不存在或已退出
        print("没有正在运行的 SGLang 子进程。")  # 无需处理
    else:
        try:
            os.killpg(os.getpgid(_sglang_process.pid), signal.SIGTERM)  # 终止进程组
        except ProcessLookupError:
            _sglang_process.terminate()  # 回退到直接终止
        _sglang_process.wait(timeout=30)  # 等待进程退出
        print("SGLang 服务已停止，GPU 显存已释放。")  # 确认完成

## 常见报错与排查

### 1）未观察到明显加速 / 前缀缓存似乎无效

**原因**：
- 模型太小（0.5B），前缀计算本身很快，缓存命中的收益不明显
- 系统提示词太短，缓存节省的计算量有限
- 首次请求的 KV 缓存尚未完成写入就发送了第二次请求

**解决**：
- 换用更大的模型（如 7B 级别），前缀缓存收益会更显著
- 使用更长的系统提示词（数百至上千字符）
- 确保请求之间有短暂间隔
- 增加测试轮数取平均值

### 2）服务返回 500 错误 / 内部服务器错误

**原因**：模型加载不完整或请求参数不兼容。

**解决**：
- 在终端手动运行启动命令查看完整错误日志
- 确认模型与 SGLang 版本兼容（查阅 SGLang 支持的模型列表）
- 检查 `max_tokens` 是否超过模型的最大上下文长度
- 尝试重启服务

---

**结语**：RadixAttention 是 SGLang 区别于其他推理框架的重要特性。在实际生产中，当大量请求共享相同的系统提示词或对话前缀时，RadixAttention 能显著降低整体推理延迟和计算成本。建议在更大模型和更长前缀场景中进一步测试其性能收益。